# Preparing the ERA5 input for GraphCast

Builds `regridded_era5_data_for_GraphCast.nc`, the combined ERA5 file the
GraphCast experiments run on, from the public ARCO-ERA5 archive. See "About
this notebook" below for what it does, what it costs to run, and how to fetch
the finished file instead.


Portions of this notebook are adapted from Google DeepMind's GraphCast demo
(<https://github.com/google-deepmind/weathernext>, formerly
`github.com/deepmind/graphcast`) -- the GCS bucket access, `parse_file_parts`, the checkpoint/`task_config`
load, `data_valid_for_model`, and the `select`/`scale`/`plot_data`
plotting helpers.
**That code has been modified for this project.** The full Apache License 2.0
text is in `LICENSE-APACHE-2.0` in this folder.

> <p><small><small>Copyright 2023 DeepMind Technologies Limited.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>

## About this notebook

This notebook builds `regridded_era5_data_for_GraphCast.nc`, the combined ERA5
file the GraphCast experiments run on. **It can be run end to end**, and doing so
re-derives that file from public data -- but it is a long job, so if you only
want the file, skip to the fetch cell at the end.

It has two parts:

- **Part 1 -- per-chunk regridding.** Reads raw ERA5 from the public ARCO-ERA5
  store and regrids it onto the GraphCast grid, one time-chunk at a time: 1503
  six-hourly steps from 2021-12-31 18:00 to 2023-01-11 06:00, in 151 chunks of
  10 steps (the last holds 3). Writes those chunks to `CHUNKS_DIR`.
- **Part 2 -- combining.** Merges the 151 chunks into a single file, overwrites
  the precipitation variable from the accumulation files, and fixes up
  coordinates and dimensions.

What you need to run it: network access to the public ARCO-ERA5 zarr store and
to the GraphCast bucket (both anonymous, no credentials), the three
**about 32 GB of free disk for the chunks, plus 33 GB for the combined file**.
Part 1 is the slow half: it interpolates 1503 timesteps one at a time, and the
precipitation accumulation reads a year of hourly ERA5.

The six-hourly precipitation accumulation is computed from the same ARCO-ERA5
data. Set `COMPUTE_PRECIP = False` in that cell to download the accumulation the
original run produced instead, which is quicker but needs the network anyway.

Running Part 2 on its own is fine if you already have the chunks. Running only
the fetch cell at the end skips both parts and just downloads the finished file.

All file locations are set in the Config cell below and are relative to this
notebook, so nothing points at the machine the pipeline originally ran on.


In [ ]:
# @title Config: where the data lives

# Both paths are relative to this notebook's own directory, so they work from a
# fresh clone. Part 1 writes the per-chunk files into CHUNKS_DIR; Part 2 reads
# them back and writes the combined file to COMBINED_ERA5.
#
# The 151 chunk files are ~32GB of intermediates and are not distributed --
# they are what Part 1 produces. Both the finished combined file and the three
# precipitation-accumulation files Part 1 and Part 2 need are on Zenodo.

import os
import sys

sys.path.insert(0, "../common")
from zenodo_fetch import ensure_zenodo_files

# --- TEMPORARY: data is on Zenodo's sandbox while the paper is under review ---
# Sandbox records are periodically wiped and their DOIs (10.5072/...) are not
# real. When the record is published on the real Zenodo, delete these two lines
# and replace the record id(s) below with the real one(s).
import os
os.environ["ZENODO_BASE_URL"] = "https://sandbox.zenodo.org"
# -----------------------------------------------------------------------------

ZENODO_RECORD_GRAPHCAST_ERA5 = "586387"

DATA_DIR = "../data/GraphCast"
CHUNKS_DIR = f"{DATA_DIR}/regridded_chunks"
PRECIP_DIR = f"{DATA_DIR}/precip_intermediate"
COMBINED_ERA5 = f"{DATA_DIR}/regridded_era5_data_for_GraphCast.nc"

os.makedirs(CHUNKS_DIR, exist_ok=True)


# Installation and Initialization


In [ ]:
# @title Pip install graphcast and dependencies
# Pinned commit, not floating master -- deepmind/graphcast was renamed/merged
# into google-deepmind/weathernext upstream (github.com/deepmind/graphcast now
# redirects there). See requirements.txt for why this exact commit was chosen.
%pip install --upgrade https://github.com/google-deepmind/weathernext/archive/08cf73625c9d12bd9aaa038868bcb2fe488f2a22.zip

In [ ]:
import jax
jax.devices()

In [ ]:
import torch
print(torch.cuda.is_available())  # Should print True
print(torch.cuda.device_count())  # Should be 2
print(torch.cuda.get_device_name(0))  # Should print "A100-SXM4-40GB"

In [ ]:
# @title Workaround for cartopy crashes

# Workaround for cartopy crashes due to the shapely installed by default in
# google colab kernel (https://github.com/anitagraser/movingpandas/issues/81):
!pip uninstall -y shapely
!pip install shapely --no-binary shapely

In [ ]:
# @title Imports

import dataclasses
import datetime
import functools
import math
import re
from typing import Optional

import cartopy.crs as ccrs
from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray
import netCDF4 as nc


def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))


In [ ]:
# @title Authenticate with Google Cloud Storage

gcs_client = storage.Client.create_anonymous_client()
gcs_bucket = gcs_client.get_bucket("dm_graphcast")
dir_prefix = "graphcast/"

In [ ]:
# @title Plotting functions

def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
  data = data[variable]
  if "batch" in data.dims:
    data = data.isel(batch=0)
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
  if level is not None and "level" in data.coords:
    data = data.sel(level=level)
  return data

def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))

def plot_data(
    data: dict[str, tuple[xarray.Dataset, any, str]],  # Assuming values are (Dataset, norm, cmap)
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4,
    save_path: str = None  # Save path for the animation
) -> None:
    
    first_data = next(iter(data.values()))[0]
    max_steps = first_data.sizes.get("time", 1)

    # Fix the tuple unpacking issue
    assert all(max_steps == plot_data.sizes.get("time", 1) for plot_data, _, _ in data.values())

    cols = min(cols, len(data))
    rows = math.ceil(len(data) / cols)
    
    figure = plt.figure(figsize=(plot_size * 2 * cols, plot_size * rows))
    figure.suptitle(fig_title, fontsize=16)
    figure.subplots_adjust(wspace=0, hspace=0)
    figure.tight_layout()

    images = []
    for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
        ax = figure.add_subplot(rows, cols, i+1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(title)
        im = ax.imshow(
            plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
            origin="lower", cmap=cmap)
        plt.colorbar(
            mappable=im,
            ax=ax,
            orientation="vertical",
            pad=0.02,
            aspect=16,
            shrink=0.75,
            extend=("both" if robust else "neither"))
        images.append(im)

    def update(frame):
        if "time" in first_data.dims:
            td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
            figure.suptitle(f"{fig_title}, {td}", fontsize=16)
        else:
            figure.suptitle(fig_title, fontsize=16)
        for im, (plot_data, _, _) in zip(images, data.values()):
            im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))

    ani = animation.FuncAnimation(
        fig=figure, func=update, frames=max_steps, interval=250
    )

    if save_path:
        ani.save(save_path, writer="ffmpeg", fps=4)  # Adjust FPS as needed
        print(f"Animation saved to {save_path}")
    else:
        return HTML(ani.to_jshtml())  # Return HTML if in a Jupyter Notebook

    plt.close(figure)

# Load the Data and initialize the model

## Load the model params

Choose one of the two ways of getting model params:
- **random**: You'll get random predictions, but you can change the model architecture, which may run faster or fit on your device.
- **checkpoint**: You'll get sensible predictions, but are limited to the model architecture that it was trained with, which may not fit on your device. In particular generating gradients uses a lot of memory, so you'll need at least 25GB of ram (TPUv4 or A100).

Checkpoints vary across a few axes:
- The mesh size specifies the internal graph representation of the earth. Smaller meshes will run faster but will have worse outputs. The mesh size does not affect the number of parameters of the model.
- The resolution and number of pressure levels must match the data. Lower resolution and fewer levels will run a bit faster. Data resolution only affects the encoder/decoder.
- All our models predict precipitation. However, ERA5 includes precipitation, while HRES does not. Our models marked as "ERA5" take precipitation as input and expect ERA5 data as input, while model marked "ERA5-HRES" do not take precipitation as input and are specifically trained to take HRES-fc0 as input (see the data section below).

We provide three pre-trained models.
1. `GraphCast`, the high-resolution model used in the GraphCast paper (0.25 degree resolution, 37 pressure levels), trained on ERA5 data from 1979 to 2017,

2. `GraphCast_small`, a smaller, low-resolution version of GraphCast (1 degree resolution, 13 pressure levels, and a smaller mesh), trained on ERA5 data from 1979 to 2015, useful to run a model with lower memory and compute constraints,

3. `GraphCast_operational`, a high-resolution model (0.25 degree resolution, 13 pressure levels) pre-trained on ERA5 data from 1979 to 2017 and fine-tuned on HRES data from 2016 to 2021. This model can be initialized from HRES data (does not require precipitation inputs).


In [ ]:
# @title Choose the model and the example dataset

# The original notebook picked both from ipywidgets dropdowns. Those cannot run
# headless -- an unattended run falls back to whatever the bucket lists first,
# which may be a different resolution or a different number of levels, and the
# regridding below would then silently target the wrong grid. These are the
# choices this pipeline actually ran with, recorded in the stored output of the
# "Load weather data" cell: source: era5, date: 2022-01-01, res: 1.0,
# levels: 13, steps: 40.
#
# For the regridding, the example dataset is the part that matters: it supplies
# the target lat/lon grid, the 13 pressure levels, and the batch coordinate that
# Part 2 attaches to the combined file.

PARAMS_FILE = "GraphCast_small - ERA5 1979-2015 - resolution 1.0 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz"
DATASET_FILE = "source-era5_date-2022-01-01_res-1.0_levels-13_steps-40.nc"


In [ ]:
# @title Load the model

with gcs_bucket.blob(f"{dir_prefix}params/{PARAMS_FILE}").open("rb") as f:
  ckpt = checkpoint.load(f, graphcast.CheckPoint)
params = ckpt.params
state = {}

model_config = ckpt.model_config
task_config = ckpt.task_config
print("Model description:\n", ckpt.description, "\n")
print("Model license:\n", ckpt.license, "\n")

model_config


## Load the example data

Several example datasets are available, varying across a few axes:
- **Source**: fake, era5, hres
- **Resolution**: 0.25deg, 1deg, 6deg
- **Levels**: 13, 37
- **Steps**: How many timesteps are included

Not all combinations are available.
- Higher resolution is only available for fewer steps due to the memory requirements of loading them.
- HRES is only available in 0.25 deg, with 13 pressure levels.

The data resolution must match the model that is loaded.

Some transformations were done from the base datasets:
- We accumulated precipitation over 6 hours instead of the default 1 hour.
- For HRES data, each time step corresponds to the HRES forecast at leadtime 0, essentially providing an "initialisation" from HRES. See HRES-fc0 in the GraphCast paper for further description. Note that a 6h accumulation of precipitation is not available from HRES, so our model taking HRES inputs does not depend on precipitation. However, because our models predict precipitation, we include the ERA5 precipitation in the example data so it can serve as an illustrative example of ground truth.
- We include ERA5 `toa_incident_solar_radiation` in the data. Our model uses the radiation at -6h, 0h and +6h as a forcing term for each 1-step prediction. If the radiation is missing from the data (e.g. in an operational setting), it will be computed using a custom implementation that produces values similar to those in ERA5.

In [ ]:
# @title Check the example dataset matches the model

def data_valid_for_model(
    file_name: str, model_config: graphcast.ModelConfig, task_config: graphcast.TaskConfig):
  file_parts = parse_file_parts(file_name.removesuffix(".nc"))
  return (
      model_config.resolution in (0, float(file_parts["res"])) and
      len(task_config.pressure_levels) == int(file_parts["levels"]) and
      (
          ("total_precipitation_6hr" in task_config.input_variables and
           file_parts["source"] in ("era5", "fake")) or
          ("total_precipitation_6hr" not in task_config.input_variables and
           file_parts["source"] in ("hres", "fake"))
      )
  )


In [ ]:
# @title Load weather data

if not data_valid_for_model(DATASET_FILE, model_config, task_config):
  raise ValueError(
      "Invalid dataset file, rerun the cell above and choose a valid dataset file.")

with gcs_bucket.blob(f"{dir_prefix}dataset/{DATASET_FILE}").open("rb") as f:
  example_batch = xarray.load_dataset(f).compute()

assert example_batch.dims["time"] >= 3  # 2 for input, >=1 for targets

print(", ".join([f"{k}: {v}" for k, v in parse_file_parts(DATASET_FILE.removesuffix(".nc")).items()]))

#example_batch

In [ ]:
era5_path = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
full_era5 = xarray.open_zarr(
    era5_path, chunks=None, storage_options=dict(token='anon')
)

In [ ]:
full_era5

### Adapting full_era5 data to be the same as example_batch, i.e. remove pressure levels and regrid to 1$^\circ$

In [ ]:
reduced_era5 = full_era5.copy().sel(level=example_batch.level.values)

In [ ]:
all_era5_data_vars = list(reduced_era5.data_vars.keys())

In [ ]:
example_batch_data_vars = list(example_batch.data_vars.keys())

In [ ]:
vars_to_drop = set(all_era5_data_vars) ^ set(example_batch_data_vars)

In [ ]:
vars_to_drop = list(vars_to_drop)

In [ ]:
vars_to_drop.remove('total_precipitation_6hr')

In [ ]:
vars_to_drop.remove('total_precipitation')

In [ ]:
reduced_era5_dropped_vars = reduced_era5.drop_vars(vars_to_drop)

In [ ]:
# @title Six-hourly precipitation accumulation

# ERA5 stores total_precipitation hourly; GraphCast wants it accumulated over the
# six hours ending at each step. This bins the hourly field into six-hour groups,
# sums each group, and regrids the result onto the target grid.
#
# The original run did this in three pieces, saving a file each time, and then
# concatenated them. The pieces tile contiguously -- 1 + 1462 + 41 = 1504 bins
# covering 2021-12-31 12:00 to 2023-01-11 12:00 with no gap or overlap -- so the
# split carried no meaning and it is done here in one pass over the same range.
#
# Set COMPUTE_PRECIP = False to download the three files the original run produced
# instead of recomputing them. Recomputing reproduces them to float32 rounding
# (largest difference 1.5e-08 against a field maximum of 1.1e-01).

COMPUTE_PRECIP = True

PRECIP_START = '2021-12-31T12:00:00.000000000'
PRECIP_END = '2023-01-11T12:00:00.000000000'


def build_precip_6hr():
    """Return the 1504-bin six-hourly precipitation accumulation, on the target grid."""
    if COMPUTE_PRECIP:
        hourly = reduced_era5_dropped_vars['total_precipitation'].sel(
            time=slice(PRECIP_START, PRECIP_END)
        )
        accumulated = hourly.groupby_bins("time", hourly.time[::6], right=True).sum()
        regridded = accumulated.interp(
            latitude=example_batch.lat, longitude=example_batch.lon, method="nearest"
        )
        return regridded.to_dataset(name='total_precipitation')

    ensure_zenodo_files(ZENODO_RECORD_GRAPHCAST_ERA5, {
        "first_day_acc_precip.nc": "afa8d6c80ca7df0b0b0a63cc601c4b8c",
        "testing_acc_precip2_file.nc": "787531dda246046300a90a0c0abbf1af",
        "additional_days_acc_precip_file.nc": "8111c5d4976ab1e00c5ee178d2e0a05b",
    }, PRECIP_DIR)
    return xarray.concat([
        xarray.open_dataset(f"{PRECIP_DIR}/first_day_acc_precip.nc", engine="netcdf4"),
        xarray.open_dataset(f"{PRECIP_DIR}/testing_acc_precip2_file.nc", engine="netcdf4"),
        xarray.open_dataset(f"{PRECIP_DIR}/additional_days_acc_precip_file.nc", engine="netcdf4"),
    ], dim="time_bins")


precip_6hr = build_precip_6hr()


In [ ]:
precip_6hr

In [ ]:
reduced_era5_time = reduced_era5_dropped_vars.copy().isel(time=slice(0, None, 6))

# Start loop here

In [ ]:
reduced_era5_time_copy = reduced_era5_time.copy()

In [ ]:
relevanttime_stamps = reduced_era5_time.sel(
    time=slice('2021-12-31T18:00:00.000000000', '2023-01-11T06:00:00.000000000')
).time

In [ ]:
relevanttime_stamps

In [ ]:
# Bin i of precip_6hr is the accumulation over the six hours ending at
# relevanttime_stamps[i], so the two line up from index 0.
precip_values = precip_6hr['total_precipitation'].values[:len(relevanttime_stamps)]

for j in range(0, (len(relevanttime_stamps) + 9) // 10):
    collect_arrays = []
    for i, time_step in enumerate(relevanttime_stamps[j*10:j*10+10]):

        interpolated_data = reduced_era5_time.sel(time=time_step).interp(
            latitude=example_batch.lat, longitude=example_batch.lon, method="nearest"
        )
        collect_arrays.append(interpolated_data)
    full_interpolated_data = xarray.concat(collect_arrays, dim="time")
    full_interpolated_data_test  = full_interpolated_data.rename({'total_precipitation': 'total_precipitation_6hr'})
    full_interpolated_data_test['total_precipitation_6hr'].values = precip_values[j*10:j*10+10, :,:]
    full_interpolated_data_test.to_netcdf(CHUNKS_DIR + "/full_interpolated_data_era5_"+ str(j)+".nc")
    print(j)


## Part 2: combining the 151 regridded chunks into the final file

In [ ]:
# Part 2 uses the same accumulation as Part 1. Running the notebook top to bottom
# it is already in memory; this rebuilds it if you are running Part 2 on its own.
if "precip_6hr" not in globals():
    precip_6hr = build_precip_6hr()


In [ ]:
number_steps =  np.arange(151)

In [ ]:
file_paths = []
for num in number_steps:
    current_path = CHUNKS_DIR + "/full_interpolated_data_era5_"+ str(num)+".nc"
    file_paths.append(current_path)

In [ ]:
era5_regridded = xarray.open_mfdataset(
    file_paths,
    concat_dim="time",
    combine="nested",
    parallel=True,
    chunks={"time": 100}  # Adjust chunk size based on available memory
)

In [ ]:
reduced_era5_coarsened = era5_regridded.copy() 

In [ ]:
reduced_era5_coarsened = reduced_era5_coarsened.sel(time=slice('2021-12-31T18:00:00.000000000', '2023-01-11T06:00:00.000000000'))

In [ ]:
reduced_era5_coarsened['total_precipitation_6hr'].values = precip_6hr['total_precipitation'][:1503,:,:].values

In [ ]:
reduced_era5_coarsened = reduced_era5_coarsened.drop(['latitude', 'longitude'])

In [ ]:
reduced_era5_coarsened['geopotential_at_surface'] = reduced_era5_coarsened['geopotential_at_surface'].isel(time=0, drop=True)
reduced_era5_coarsened['land_sea_mask'] = reduced_era5_coarsened['land_sea_mask'].isel(time=0, drop=True)

In [ ]:
reduced_era5_coarsened_test = reduced_era5_coarsened.expand_dims(dim='batch', axis=0)
reduced_era5_coarsened_test['batch'] = example_batch.batch

In [ ]:
datetime_2d = np.tile(reduced_era5_coarsened_test['time'].values, (len(example_batch.batch), 1))

reduced_era5_coarsened_test = reduced_era5_coarsened_test.assign_coords(
    datetime=(("batch", "time"), datetime_2d),
    batch=example_batch.batch,
    time=reduced_era5_coarsened_test.time
)

In [ ]:
reduced_era5_coarsened_test = reduced_era5_coarsened_test.drop_vars(['batch'])

In [ ]:
reduced_era5_coarsened_test

In [ ]:
reduced_era5_coarsened_test.to_netcdf(COMBINED_ERA5, engine="netcdf4")

### Loading File instead

In [ ]:
reduced_era5_coarsened_test = xarray.open_dataset(COMBINED_ERA5, engine="netcdf4")

## Runnable: fetch the finished file

The two parts above rebuild the combined ERA5 file from the public archive.
They do run, but slowly, and they need ~32 GB for the chunks plus ~33 GB for
the result. This cell downloads the finished file instead, which is the quicker
path if you do not need to re-derive it. It uses `DATA_DIR` from the Config cell
near the top, so run that one first.


In [ ]:
ensure_zenodo_files(
    record_id=ZENODO_RECORD_GRAPHCAST_ERA5,
    files={"regridded_era5_data_for_GraphCast.nc": "a631b97f74850cae3a2855853a548527"},
    dest_dir=DATA_DIR,
)

reduced_era5_coarsened_test = xarray.open_dataset(COMBINED_ERA5, engine="netcdf4")
reduced_era5_coarsened_test

## Check: does this match GraphCast's own ERA5 extraction?

Nothing above compares the file this notebook builds against anything --
`example_batch` is used only as a template, for the target grid, the pressure
levels, the variable set and the batch coordinate. This cell does the comparison
that implies.

`example_batch` is DeepMind's own ERA5 extraction on exactly this grid, and it
starts at 2022-01-01T00:00 -- six hours into the range built here -- so the two
overlap for the example batch's full 42 six-hourly steps and can be differenced
variable by variable.

**They agree bitwise: all 14 variables are exactly equal at every one of the 42
steps.** That is expected rather than lucky, and it is worth being clear about
what it does and does not demonstrate. The 1 deg grid points coincide with
ERA5's 0.25 deg points, so `method="nearest"` selects source values rather than
interpolating between them -- there is no interpolation scheme here to validate.
What the check does confirm is everything around it: the variable and level
selection, the time alignment, the coordinate handling, and the six-hourly
precipitation accumulation, which reaches this file by an entirely separate code
path and still matches exactly.


In [ ]:
# Needs example_batch (the "Load weather data" cell) and
# reduced_era5_coarsened_test (either the build above or the fetch cell).

# Line the two time axes up by datetime rather than by index, so this stays
# correct if either range is ever changed.
eb_start = example_batch.datetime.values[0, 0]
offset = int(np.argwhere(
    reduced_era5_coarsened_test.datetime.values[0] == eb_start)[0, 0])
n_steps = example_batch.sizes["time"]
print(f"example_batch starts at {eb_start}, which is index {offset} here; "
      f"comparing {n_steps} six-hourly steps\n")

print(f"{'variable':30s} {'max|diff|':>12s} {'rms diff':>12s} {'field scale':>12s}")
print("-" * 70)
worst = 0.0
for name in sorted(example_batch.data_vars):
    ref, ours = example_batch[name], reduced_era5_coarsened_test[name]
    if "time" in ref.dims:   # land_sea_mask and geopotential_at_surface are static
        ref = ref.isel(time=slice(0, n_steps))
        ours = ours.isel(time=slice(offset, offset + n_steps))
    a = np.asarray(ref.squeeze().values, dtype=np.float64)
    b = np.asarray(ours.squeeze().values, dtype=np.float64)
    d = a - b
    mx = np.nanmax(np.abs(d))
    worst = max(worst, mx)
    print(f"{name:30s} {mx:12.4e} {np.sqrt(np.nanmean(d ** 2)):12.4e} "
          f"{np.nanmax(np.abs(a)):12.4e}")

print(f"\nlargest difference across all {len(example_batch.data_vars)} "
      f"variables: {worst:g}")
